In [1]:
from collections import defaultdict
import numpy as np
import pandas as pd
import random
import re
from tqdm import tqdm
import torch
from transformers import LlamaTokenizer, LlamaForCausalLM, LlamaModel
from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaTokenizer, Qwen2ForCausalLM, Qwen2Tokenizer

import os
import pickle
import seaborn as sns
from copy import deepcopy

os.environ["REQUESTS_CA_BUNDLE"] = "/etc/ssl/certs/ca-certificates.crt"
os.environ["SSL_CERT_FILE"] = "/etc/ssl/certs/ca-certificates.crt"

import random
import json
import matplotlib.pyplot as plt
import seaborn as sbn
from sklearn.metrics import balanced_accuracy_score

device = "cuda:1"
STORAGE_DIR = "cache/"

In [ ]:
# Loaders for LLaMA models

# Change this cell according to the model you use !!!!
model = AutoModelForCausalLM.from_pretrained("MODEL_NAME",
                                         cache_dir="/mnt/hdd_drive/huggingface/hub/",
                                         torch_dtype=torch.float16#,
                                        )
tokenizer = AutoTokenizer.from_pretrained("MODEL_NAME",
                                            cache_dir="/mnt/hdd_drive/huggingface/hub/"
                                        )
model = model.to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
LAYER_NUM = model.config.num_hidden_layers
HEAD_NUM = model.config.num_attention_heads

HEAD_SPAN = model.config.hidden_size // model.config.num_attention_heads
MODEL_NAME = model.config._name_or_path.split("/")[-1]
# !! CHANGE the following line if you use this code with models implementing Grouped query attention
QUERIES_PER_KEY = model.config.num_attention_heads // model.config.num_key_value_heads

print(LAYER_NUM, HEAD_NUM, HEAD_SPAN, QUERIES_PER_KEY, MODEL_NAME)

LETTER_TOKENS = np.array([tokenizer(x)['input_ids'][-1] for x in ["false", "true"]])
print(LETTER_TOKENS)
LETTER_TOKENS2 = np.array([tokenizer(x)['input_ids'][-1] for x in ["False", "True"]])
print(LETTER_TOKENS2)

In [5]:
from collections import OrderedDict 

class NewModel(torch.nn.Module):
    def __init__(self, model, *args):
        super().__init__(*args)
        self.selected_out = OrderedDict()

        self.pretrained = model
        self.fhooks = []

        for i in range(LAYER_NUM):
            self.fhooks.append(self.pretrained.model.layers[i].self_attn.q_proj
                .register_forward_hook(self.forward_hook("query_vec_" + str(i))))
            self.fhooks.append(self.pretrained.model.layers[i].self_attn.k_proj
                .register_forward_hook(self.forward_hook("key_vec_" + str(i))))
    
    def forward_hook(self, layer_name):
        def hook(module, input, output):
            self.selected_out[layer_name] = output.cpu()
        return hook

    def forward(self, x):        
        out = self.pretrained(**x)
        return out, self.selected_out
    
newmodel = NewModel(model)

In [6]:
def angular_dist(vec_a, vec_b):
    # Without normalization it works much faster and yields better results for some unknown reason
    return torch.sum(vec_a * vec_b) #/ (vec_a.norm(p=2) * vec_b.norm(p=2))
    
def get_negation(sss):
    separator = ''
    if len(sss) == 0:
        return ''
    if sss.find(' and ') != -1:
        pos = sss.find(' and ')
        parts = ['', '']
        separator = ' or '
        parts[0] = get_negation(sss[:pos])
        parts[1] = get_negation(sss[pos + 5:])
    elif sss.find(' or ') != -1:
        pos = sss.find(' or ')
        separator = ' and '
        parts = ['', '']
        parts[0] = get_negation(sss[:pos])
        parts[1] = get_negation(sss[pos + 4:])
    elif sss.find(',') != -1:
        separator = ', '
        parts = ['a', 'b']
        parts[0] = get_negation(sss[:sss.find(',')])
        parts[1] = get_negation(sss[sss.find(',') + 2:])
        if parts[1] == '':
            separator = ','
    else:
        if sss.find(' not ') != -1:
            parts = sss.split(' not ')
            return parts[0] + ' ' + parts[1]
        if sss[:4] == 'not ':
            return sss[4:]
        if sss.find(' is ') != -1:
            parts = sss.split(' is ')
            return parts[0] + ' is not ' + parts[1]
        if sss.find(' a ') != -1:
            parts = sss.split(' a ')
            return parts[0] + ' not a ' + parts[1]
        return 'not ' + sss

    result = parts[0] + separator + parts[1]

    if result.find(' or ') != -1:
        result = result.replace(', ', ' or ')
        result = result.replace(' or or ', ' or ')
        
    if result.find(' and ') != -1:
        result = result.replace(' and ', ', ')
        result = result.replace(',,', ',')

        for i in range(len(result) -1, 0, -1):
            if result[i] == ',':
                if result[:i].find(',') != -1:
                    result = result[:i] + ', and ' + result[i + 2:]
                else:
                    result = result[:i] + ' and ' + result[i + 2:]
                break
    return result

In [7]:
def get_negative_samples(data, rg = range(10000)):
    true_labels = []

    for EXMPL in rg:
        for XXX in range(len(data["example" + str(EXMPL)])-1):
            elem = data["example" + str(EXMPL)]["in_context_example" + str(XXX)]
            query = elem['query'].split(': ')[1]

            if query.find(" not ") != -1:
                true_labels.append(0)
            else:
                true_labels.append(1)

    return np.array(true_labels)

In [ ]:
def do_calc_with_neg(data, rg=range(10000), no_context=False, shuffle=False, max_distractor=-1, mask=[]):
    true_labels = []
    predicted_labels = []
    predicted_raw = []
    next_token_labels = []

    for LAYER in range(LAYER_NUM):
        predicted_labels.append([])
        predicted_raw.append([])

        for HEAD in range(HEAD_NUM):
            predicted_labels[LAYER].append([])
            predicted_raw[LAYER].append([])
    
    EXMPL_NUM = -1
    for EXMPL in rg:
        EXMPL_NUM += 1
        for XXX in range(len(data["example" + str(EXMPL)])-1):
            if XXX + EXMPL_NUM * 8 not in mask:
                continue
            
            for code in range(2):
                elem = data["example" + str(EXMPL)]["in_context_example" + str(XXX)]
                context = elem['question']
                query = elem['query'].split(': ')[1]
                
                if code == 0:
                #    print(query)
                    query = get_negation(query)
               #     print(query)
                if max_distractor > 0:
                    parts = (context + ' ').split('. ')[:-1]
                    parts2 = []
                    cnt = 0
                    for tmp in parts:
                        if tmp + '.' not in elem["chain_of_thought"] and get_negation(tmp + '.') not in elem["chain_of_thought"] \
                        and get_negation(get_negation(tmp + '.')) not in elem["chain_of_thought"]:
                            if cnt < max_distractor:
                                cnt += 1
                                parts2.append(tmp)
                        else:
                            parts2.append(tmp)
                    context = '. '.join(parts2) + '.'
                    
                if shuffle:
                    parts = (context + ' ').split('. ')[:-1]
                    np.random.shuffle(parts)
                    context = '. '.join(parts) + '.'
                if no_context:
                    text = "Answer whether the Statement is true or false.\nStatement: " + query + "\n"
                else:
                    text = "Use provided context and answer whether the statement is true or false.\nContext: " + context  + "\nStatement: " + query + "\n"
                
                encodinds_context_q = tokenizer(text, return_tensors="pt")
                num_q = encodinds_context_q["input_ids"].shape[-1] - 1
                true_labels.append(code)
                        
                predicts = np.zeros((LAYER_NUM, HEAD_NUM, 2))        
                for i, option in enumerate(["false", "true"]):           
                    encodings_answ = tokenizer(text + "Answer: " + option, return_tensors="pt")
                    with torch.no_grad():
                        outputs = newmodel(encodings_answ.to(device))
        
                    for LAYER in range(LAYER_NUM):
                        for HEAD in range(HEAD_NUM):
                            predicts[LAYER][HEAD][i] = angular_dist(outputs[1]["query_vec_" + str(LAYER)][0][-1][HEAD * HEAD_SPAN:(HEAD + 1) * HEAD_SPAN], 
                                                                                                outputs[1]["key_vec_" + str(LAYER)][0][num_q][(HEAD // QUERIES_PER_KEY)  * HEAD_SPAN:((HEAD // QUERIES_PER_KEY) + 1) * HEAD_SPAN])
                for LAYER in range(LAYER_NUM):
                    for HEAD in range(HEAD_NUM):
                        predicted_labels[LAYER][HEAD].append(np.argmax(predicts[LAYER][HEAD]))
                        predicted_raw[LAYER][HEAD].append(predicts[LAYER][HEAD])
                logits = outputs[0].logits.detach().cpu()
                logits = logits[:, -2, :]
                logits_full = logits.squeeze(0)
                next_token_raw = np.argmax(logits_full)
                logits_reduced = logits_full[LETTER_TOKENS].numpy() + logits_full[LETTER_TOKENS2].numpy()
        
                next_token_labels.append(np.argmax(logits_reduced))

    for LAYER in range(LAYER_NUM):
        predicted_labels[LAYER] = np.array(predicted_labels[LAYER])
        predicted_raw[LAYER] = np.array(predicted_raw[LAYER])

    predicted_labels = np.stack(predicted_labels)
    predicted_raw = np.stack(predicted_raw)
    next_token_labels = np.array(next_token_labels)
    true_labels = np.array(true_labels)
    
    return true_labels, (predicted_labels, predicted_raw), next_token_labels

In [12]:
def get_results(true_labels, qk_predicts, baseline, mask_train=None):   
    mn = np.zeros((LAYER_NUM,HEAD_NUM))
    for i in range(LAYER_NUM):
        for j in range(HEAD_NUM):
            mn[i][j] = balanced_accuracy_score(true_labels[mask_train], qk_predicts[i][j][mask_train]) 

    best_head = (np.argmax(mn) // HEAD_NUM, np.argmax(mn) % HEAD_NUM)
    #baseline_res = balanced_accuracy_score(true_labels, baseline)
    print("best head ", np.argmax(mn) // HEAD_NUM, np.argmax(mn) % HEAD_NUM)
    #print("Baseline score", baseline_res)
    return mn, best_head

In [ ]:
def call_noise(file_name, idx_rg, max_distractor=-1, shuffle=True, draw_matrix=True, heatmap_name="Some name"):
    with open(file_name, 'r') as file:
        data = json.load(file)

    np.random.seed(42)
    # Numeration of samples in data file is 1-based
    idx_rg = np.array(idx_rg) + 1
    negative_samples = get_negative_samples(data, idx_rg)

    not_ratio = np.mean(negative_samples) 
    if not_ratio < 0.1 or not_ratio > 0.9:
        print("Classes are too unbalanced: ", not_ratio)
        return -1
    negatives = np.sum(1 - negative_samples)        
    positives = np.sum(negative_samples)

    samples_to_select = min(negatives, positives)
    
    np.random.seed(1337)
    mask = np.hstack([np.random.choice(np.arange(len(negative_samples))[negative_samples == 0], samples_to_select, replace=False),
        np.random.choice(np.arange(len(negative_samples))[negative_samples == 1], samples_to_select, replace=False)])

    print("Results are based on " + str(len(mask)) + " samples. Calibration set - 600")
    true_labels2, (predicted_labels_2, predicted_raw), next_token_labels_2 = do_calc_with_neg(data, idx_rg, max_distractor=max_distractor,\
                                                                                              shuffle=shuffle, mask=mask)

    idx_train = np.arange(len(true_labels2))
    idx_train = np.hstack([idx_train[true_labels2 == 0][:300], idx_train[true_labels2 == 1][:300]])
    mask_train = np.zeros(len(true_labels2), dtype=bool)
    mask_train[idx_train] = True
   # print(balanced_accuracy_score(true_labels2, next_token_labels_2))
    mtrx2, best_head = get_results(true_labels2, predicted_labels_2, next_token_labels_2, mask_train=mask_train)
    baseline_acc = balanced_accuracy_score(true_labels2[~mask_train], next_token_labels_2[~mask_train])
    head_acc = balanced_accuracy_score(true_labels2[~mask_train], predicted_labels_2[best_head[0]][best_head[1]][~mask_train])
    print("Best head, test acc", head_acc)
    print("Baseline score, test acc", baseline_acc)
    if draw_matrix:
        plt.title(heatmap_name)
        sbn.heatmap(mtrx2)
        plt.show()
    return mtrx2, baseline_acc, head_acc

## Do Calculations

In [ ]:
train_mtrx = []
baseline_acc = []
best_head_acc = []

for i in range(1, 6):
    results = call_noise('data/prontoqa/{0}hop_ProofsOnly_random_nodistractor_testdistractor.json'.format(i), np.arange(500), max_distractor=-1, draw_matrix=False)
    train_mtrx.append(results[0])
    baseline_acc.append(results[1])
    best_head_acc.append(results[2])

results = {'train_mtrx': train_mtrx,
        'baseline_acc' : baseline_acc,
         'best_head_acc' : best_head_acc}

with open(STORAGE_DIR + 'tmp_{0}_mp_no_distr.pkl'.format(MODEL_NAME), 'wb') as file:
    pickle.dump(results, file)

In [ ]:
train_mtrx = []
baseline_acc = []
best_head_acc = []

for i in range(1, 6):
    results = call_noise('data/prontoqa/{0}hop_ProofsOnly_random.json'.format(i), np.arange(500), max_distractor=-1, draw_matrix=False)
    train_mtrx.append(results[0])
    baseline_acc.append(results[1])
    best_head_acc.append(results[2])

results = {'train_mtrx': train_mtrx,
        'baseline_acc' : baseline_acc,
         'best_head_acc' : best_head_acc}

with open(STORAGE_DIR + 'tmp_{0}_mp_max_distr.pkl'.format(MODEL_NAME), 'wb') as file:
    pickle.dump(results, file)

In [ ]:
train_mtrx = []
baseline_acc = []
best_head_acc = []

for i in range(1, 6):
    results = call_noise('data/prontoqa/{0}hop_Composed_2ruletypes_random.json'.format(i), np.arange(500), max_distractor=-1, draw_matrix=False)
    train_mtrx.append(results[0])
    baseline_acc.append(results[1])
    best_head_acc.append(results[2])

results = {'train_mtrx': train_mtrx,
        'baseline_acc' : baseline_acc,
         'best_head_acc' : best_head_acc}

with open(STORAGE_DIR + 'tmp_{0}_composed_max_distr.pkl'.format(MODEL_NAME), 'wb') as file:
    pickle.dump(results, file)

In [ ]:
train_mtrx = []
baseline_acc = []
best_head_acc = []

for i in range(1, 6):
    results = call_noise('data/prontoqa/{0}hop_Composed_2ruletypes_random_nodistractor_testdistractor.json'.format(i), np.arange(500), max_distractor=-1, draw_matrix=False)
    train_mtrx.append(results[0])
    baseline_acc.append(results[1])
    best_head_acc.append(results[2])

results = {'train_mtrx': train_mtrx,
        'baseline_acc' : baseline_acc,
         'best_head_acc' : best_head_acc}

with open(STORAGE_DIR + 'tmp_{0}_composed_no_distr.pkl'.format(MODEL_NAME), 'wb') as file:
    pickle.dump(results, file)

In [ ]:
train_mtrx = []
baseline_acc = []
best_head_acc = []

for i in range(1, 6):
    results = call_noise('data/prontoqa/1hop_ProofsOnly_random.json', np.arange(500), max_distractor=i, draw_matrix=False)
    train_mtrx.append(results[0])
    baseline_acc.append(results[1])
    best_head_acc.append(results[2])

results = {'train_mtrx': train_mtrx,
        'baseline_acc' : baseline_acc,
         'best_head_acc' : best_head_acc}

with open(STORAGE_DIR + 'tmp_{0}_mp_i_distr.pkl'.format(MODEL_NAME), 'wb') as file:
    pickle.dump(results, file)